# Prompting Patterns for LangChain Agents

This notebook is a practical reference for shaping agent behavior with prompts. It progresses from a baseline agent call to system instructions, few-shot examples, human-readable output structure, and finally validated structured output with Pydantic.

## Learning goals

- Establish a baseline response before changing the prompt.
- Use a system prompt to define role, scope, and behavior.
- Guide style with few-shot examples and explicit output fields.
- Return data that application code can access through a typed schema.

The examples use fictional planetary capitals so the prompting mechanics remain easy to compare. Model responses are nondeterministic, so evaluate the requested format and fields rather than expecting identical wording on every run.

# Basic Prompting

In [ ]:
# Load API keys and other settings from the repository's .env file.
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# Create a baseline agent before adding prompt instructions.
agent = create_agent(model = "gpt-5-nano")

question = HumanMessage(content = "What's the capital of the Moon?")

# Agent state is passed as a dictionary containing a list of messages.
response = agent.invoke({
    "messages": [question]
})

# The last message is the agent's final user-facing response.
print(response["messages"][-1].content)

There isn’t one. The Moon isn’t a country or government, so it has no capital. If you’re thinking of fiction, some stories invent lunar cities with capitals, but in reality there’s no capital on the Moon.


In [ ]:
# A system prompt sets behavior that should apply across user messages.
system_prompt = "You are a science fiction writer, create a capital city at the users request."

scifi_agent = create_agent(
    model = "gpt-5-nano",
    system_prompt = system_prompt
)

response = scifi_agent.invoke({
    "messages": [question]
})

# Select the final message instead of relying on a fixed message position.
print(response["messages"][-1].content)

In this sci‑fi setting, the Moon’s capital is Selene Prime.

- Location: Built on the rim of Shackleton Crater in the lunar south, Selene Prime sits where long sunlit corridors meet shadowed interiors. It’s powered by vast solar farms that line the crater wall, with ice and water reserves tucked beneath the basalt foundations.

- Government: A Lunar Confederation centralizes power in the Council of Craters, with a rotating Primarch elected by a public lottery to ensure broad representation. The city itself houses the Grand Assembly, a translucent chamber where policy and exploration quotas are debated.

- Architecture and cityscape: The surface is a lattice of glassy domes and basalt towers connected by sky-bridges and maglev walkways. Inside the crater, vertical gardens climb walls of reinforced regolith, and subterranean caverns host research labs, hydroponics, and ice-mining facilities. The centerpiece is a spiraling spire called the Helix, a symbol of lunar unity.

- Economy: Selen

## Few-Shot Examples

### Few-shot prompting

Few-shot examples show the model the relationship between an input and the desired answer. They are useful when a short instruction is not enough to communicate naming conventions, tone, or response patterns.

In [4]:
system_prompt = """ 
you are a science fiction writer, create a space capital city at the users request.

User: What is the capital of Mars?
Scifi Writer: Marsialis

User: What is the capital of Venus?
Scifi Writer: Venusovia
"""

scifi_agent = create_agent(
    model = "gpt-5-nano",
    system_prompt = system_prompt 
)

response = scifi_agent.invoke({
    "messages": [question]
})

print(response["messages"][1].content)


Lunopolis


## Structured Prompts

### Structured prompts

An explicit field list makes a response easier for a person to scan, but it is still plain text. Prompt instructions can improve consistency, yet application code must still parse the result if it needs individual values.

In [ ]:
system_prompt = """ 
You are a science fiction writer, create a space capital city
at the users request. 
Please keep to the below structure:
Name: The name of the capital city
Location: Where it is based
Vibe: 2-3 words to describe its vibe
Economy: Main industries
"""

scifi_agent_structured = create_agent(
    model = "gpt-5-nano",
    system_prompt = system_prompt
)

# Invoke the agent created for this section so the structured prompt is tested.
response = scifi_agent_structured.invoke({
    "messages": [question]
})

# The final message contains the agent's user-facing answer.
print(response["messages"][-1].content)

Lunopolis


## Structured Output

### Structured output

`response_format=CapitalInfo` asks the agent to produce data validated against a Pydantic model. This is the preferred boundary when later code needs reliable fields such as `name`, `location`, `vibe`, and `economy` instead of parsing prose.

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pydantic import BaseModel

# Define the fields the application needs, including their expected types.
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

agent = create_agent(
    model = "gpt-5-nano",
    system_prompt = "You are a science fiction writer, create a capital city at the users request.",
    # The agent validates its final answer against this schema.
    response_format = CapitalInfo
)

question = HumanMessage(content = "What is the capital of The Moon?")

response = agent.invoke({
    "messages": [question]
})

# Structured data is returned separately from the conversational messages.
response["structured_response"]

CapitalInfo(name='Lunaris Prime', location='Near-side equatorial rim of Mare Serenitatis (Sea of Serenity), accessible from Earth via direct lunar corridors', vibe='A gleaming glass-and-ceramic megacity built into a crater rim, with solar domes, orbital elevators, vertical farms, and transit tubes; a hub of lunar governance and culture', economy='Helium-3 mining, solar-energy production, advanced robotics and construction, lunar tourism, and scientific research')

In [ ]:
# Access a validated field directly instead of parsing response text.
response["structured_response"].name

'Lunaris Prime'

In [ ]:
capital_info = response["structured_response"]

# Convert validated fields into values that application code can use.
capital_name = capital_info.name
capital_location = capital_info.location

print(f"{capital_name} is located at {capital_location}")

Lunaris Prime is located at Near-side equatorial rim of Mare Serenitatis (Sea of Serenity), accessible from Earth via direct lunar corridors


## Conclusion and reuse checklist

The prompting progression in this notebook is:

1. Establish a baseline agent response.
2. Add a system prompt when the agent needs a stable role or behavior.
3. Add few-shot examples when the desired pattern is easier to demonstrate than describe.
4. Use explicit fields when the output should be readable and consistent for people.
5. Use a Pydantic response schema when downstream code needs reliable typed values.

For future projects, keep prompts close to the code that owns their behavior, test them with representative inputs, and prefer structured output at application boundaries. Change one prompting technique at a time so you can tell which instruction improved or degraded the result.